# Dynamic Travel Advisor

In [11]:
from crewai import Agent, Task, LLM, Crew
from pydantic import BaseModel, Field
from typing import List
import os
from crewai.flow.flow import Flow, start, listen, router

## Use gpt-4

In [ ]:
# Set up API key

os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"
llm = LLM(model="gpt-4o-mini")

## Agents

In [13]:
# Travel Planner Agent
planner_agent = Agent(
    llm=llm,
    role="Travel Planner",
    backstory="An expert in planning group vacations, considering budgets, destinations, and activities.",
    goal="Create a vacation plan based on traveler names, departure city, and destination.",
    verbose=True,
)

# Travel Validator Agent
validator_agent = Agent(
    llm=llm,
    role="Travel Plan Validator",
    backstory="An experienced travel advisor who ensures all plans meet safety, budget, and feasibility constraints.",
    goal="Evaluate vacation plans and validate if they meet predefined constraints.",
    verbose=True,
)

## Tasks

In [14]:
def create_vacation_plan_task(names, city, destination) -> Task:
    return Task(
        description=(
            f"Create a detailed vacation plan for the following travelers: {', '.join(names)}, "
            f"departing from {city} to {destination}. Ensure the plan includes "
            f"activities, accommodation, and estimated costs."
        ),
        expected_output="A detailed vacation plan including activities, accommodation, and costs.",
        agent=planner_agent,
    )


def validate_vacation_plan_task(vacation_plan) -> Task:
    return Task(
        description=(
            f"Evaluate the following vacation plan: {vacation_plan}. "
            f"Ensure it meets the following constraints: budget-friendly, includes at least two activities, "
            f"and has clear accommodation details. Reply with 'Valid' if it meets the constraints, "
            f"otherwise reply with 'Invalid'."
        ),
        expected_output="Valid or Invalid.",
        agent=validator_agent,
    )

## Flow

In [15]:
class TravelState(BaseModel):
    vacation_plan: str = ""
    is_plan_valid: bool = False
    generation_attempts_left: int = 2

In [18]:
class TravelAdvisorFlow(Flow[TravelState]):
    names: List[str] = Field(description="List of travelers")
    city: str = Field(description="Departure city")
    destination: str = Field(description="Dest city")

    def __init__(self, names: List[str], city: str, destination: str):
        super().__init__(names=names, city=city, destination=destination)


    @start()
    def generate_vacation_plan(self):
        print("Generating vacation plan")
        task = create_vacation_plan_task(self.names, self.city, self.destination)
        crew = Crew(agents=[planner_agent], tasks=[task])
        result = crew.kickoff()
        self.state.vacation_plan = result.raw
        print("Vacation plan generated!")

    @listen(generate_vacation_plan)
    def validate_vacation_plan(self):
        print("Start validation of the plan...")
        task = validate_vacation_plan_task(self.state.vacation_plan)
        crew = Crew(agents=[validator_agent], tasks=[task])
        result = crew.kickoff()
        self.state.is_plan_valid = "Valid" in result.raw
        print("Validation complete", "Valid" if self.state.is_plan_valid else "Invalid")

    @router(validate_vacation_plan)
    def route_vacation_plan(self):
        if self.state.is_plan_valid:
            return "valid"
        elif self.state.generation_attempts_left == 0:
            return "not_feasible"
        else:
            return "regenerate"

    @listen("valid")
    def finalize_vacation_plan(self):
        with open("vacation_plan.txt", "w") as file:
            file.write(self.state.vacation_plan)
        print("Vacation plan saved in file")

    @listen("regenerate")
    def regenerate_vacation_plan(self):
        self.state.generation_attempts_left -= 1
        self.generate_vacation_plan()

    @listen("not_feasible")
    def notify_user(self):
        print("Plan is not feasible, I'm sorry :(")


## Main

Note: `nest_asyncio` allows `asyncio.run` to work in Jupyter Notebooks by enabling nested event loops.

In [19]:
import nest_asyncio
nest_asyncio.apply()

def kickoff():
    names = ["Alice", "Robert", "Charlie"]
    city = "Berlin"
    destination = "Rome"

    travel_flow = TravelAdvisorFlow(names, city, destination)

    travel_flow.kickoff()

kickoff()

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: TravelAdvisorFlow                                                                                        │
│  ID: ae8bf4f6-5d43-414e-8be1-727fe47d9eb2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: TravelAdvisorFlow                                                                                        │
│  ID: ae8bf4f6-5d43-414e-8be1-727fe47d9eb2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: ae8bf4f6-5d43-414e-8be1-727fe47d9eb2

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: generate_vacation_plan                                                                                 │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Generating vacation plan


...

Vacation plan generated!
Start validation of the plan...


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Plan Validator                                                                                   │
│                                                                                                                 │
│  Task: Evaluate the following vacation plan: **Vacation Plan for Alice, Robert, and Charlie**                   │
│  **Departure City:** Berlin                                                                                     │
│  **Destination:** Rome                                                                                          │
│  **Duration:** 5 days                                                                                           │
│  **Travel Dates:** [Insert Specific Dates Here]                                                                 │
│  **Estimated Budget:** €1200 per person                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Day 1: Arrival in Rome                                                                                     │
│  - **Flight:**                                                                                                  │
│    - Departure: Berlin Brandenburg Airport (BER)                                                                │
│    - Arrival: Leonardo da Vinci International Airport (FCO)                                                     │
│    - Estimated Cost: €150 per person (Total €450)                                                               │
│    - Departure Time: 10:00 AM                                                                                   │
│    - Arrival Time: 12:30 PM                                                                                     │
│                                                                                                                 │
│  - **Accommodation:**                                                                                           │
│    - Hotel: Roma Luxury Suites                                                                                  │
│    - Check-in: 1:00 PM                                                                                          │
│    - Cost: €150 per night (Total for 4 nights: €600)                                                            │
│                                                                                                                 │
│  - **Evening Activity:**                                                                                        │
│    - Visit the Colosseum (Night Tour)                                                                           │
│    - Start Time: 7:00 PM                                                                                        │
│    - Duration: 2 hours                                                                                          │
│    - Estimated Cost: €25 per person (Total €75)                                                                 │
│                                                                                                                 │
│  - **Dinner:**                                                                                                  │
│    - Restaurant: Trattoria Da Enzo                                                                              │
│    - Estimated Cost: €30 per person (Total €90)        

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Plan Validator                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Current Task: Evaluate the following vacation plan: **Vacation Plan for Alice, Robert, and Charlie**           │
│  **Departure City:** Berlin                                                                                     │
│  **Destination:** Rome                                                                                          │
│  **Duration:** 5 days                                                                                           │
│  **Travel Dates:** [Insert Specific Dates Here]                                                                 │
│  **Estimated Budget:** €1200 per person                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Day 1: Arrival in Rome                                                                                     │
│  - **Flight:**                                                                                                  │
│    - Departure: Berlin Brandenburg Airport (BER)                                                                │
│    - Arrival: Leonardo da Vinci International Airport (FCO)                                                     │
│    - Estimated Cost: €150 per person (Total €450)                                                               │
│    - Departure Time: 10:00 AM                                                                                   │
│    - Arrival Time: 12:30 PM                                                                                     │
│                                                                                                                 │
│  - **Accommodation:**                                                                                           │
│    - Hotel: Roma Luxury Suites                                                                                  │
│    - Check-in: 1:00 PM                                                                                          │
│    - Cost: €150 per night (Total for 4 nights: €600)                                                            │
│                                                                                                                 │
│  - **Evening Activity:**                                                                                        │
│    - Visit the Colosseum (Night Tour)                                                                           │
│    - Start Time: 7:00 PM                                                                                        │
│    - Duration: 2 hours                                                                                          │
│    - Estimated Cost: €25 per person (Total €75)                                                                 │
│                                                                                                                 │
│  - **Dinner:**                                                                                                  │
│    - Restaurant: Trattoria Da Enzo                     

Validation complete Valid
Vacation plan saved in file


╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: TravelAdvisorFlow                                                                                        │
│  ID: ae8bf4f6-5d43-414e-8be1-727fe47d9eb2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯